# Electivo de Bioinformática — Clase 3

## Tipos de archivos en bioinformática y configuración de ambientes con conda

**Programa:** Doctorado — 2º año
**Duración:** 3 horas
**Fecha:** 1 de septiembre

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Repasar y consolidar los comandos de shell/bash vistos en las clases 1 y 2.
2. Reconocer y describir los formatos de archivo más usados en bioinformática y genómica (FASTA, FASTQ, SAM/BAM, VCF, BED/GFF/GTF) y sus archivos de índice asociados.
3. Descargar y transferir archivos genómicos entre sistemas usando `wget`, `curl`, `scp` y `rsync`, verificando su integridad.
4. Explicar qué es `conda`, por qué se usan ambientes virtuales en bioinformática y configurar los canales correctos (`bioconda` + `conda-forge`).
5. Crear un ambiente de trabajo reproducible e instalar herramientas bioinformáticas (`bwa`, `samtools`) dentro de él.
6. Dejar el ambiente y los archivos listos como punto de partida para el alineamiento de secuencias (clase 4).


---
## 1. Repaso de las clases 1 y 2 (20 min)

### Clase 1 — Conexión remota y línea de comandos

- Qué es un clúster y cómo conectarse vía **SSH**.
- Navegación básica: `pwd`, `ls`, `cd`.
- Gestión de archivos y directorios: `mkdir`, `touch`, `cp`, `mv`, `rm`.
- Lectura de archivos: `cat`, `less`, `head`, `tail`, `wc`.
- Búsqueda: `grep`, `find`.

### Clase 2 — Ambiente Linux y Bash

- Variables de shell vs. variables de entorno (`export`), `$PATH`, `alias`, `~/.bashrc`.
- Comodines (`*`, `?`, `{}`) para trabajar con lotes de archivos.
- Scripts de bash: shebang (`#!/bin/bash`), permisos (`chmod +x`), `set -euo pipefail`.
- Argumentos posicionales (`$1`, `$2`, `$#`, `$@`) y control de flujo (`if`/`elif`/`else`, `test`/`[ ]`).
- Bucles `for` y `while`, y funciones de bash.
- Procesamiento de texto con `sed` y `awk`.
- Compresión y archivado: `tar` y `gzip`.

Hoy vamos a **usar todo esto como herramienta**, no como tema central: lo aplicaremos para explorar formatos de archivo genómicos y para automatizar la instalación de programas.

In [ ]:
%%bash
# Recreamos el espacio de trabajo y creamos el de hoy
mkdir -p ~/bioinfo/clase3
cd ~/bioinfo/clase3
pwd
ls -la ~/bioinfo

---
## 2. Tipos de archivo en bioinformática y genómica (50 min)

Cada etapa de un análisis genómico produce y consume un formato de archivo distinto. Reconocerlos a simple vista (y saber qué herramienta de línea de comandos usar para mirarlos) es una habilidad tan importante como saber programar el análisis en sí.

### 2.1 Panorama general

| Formato | Extensión típica | Contiene | Se usa para |
|---|---|---|---|
| FASTA | `.fa`, `.fasta`, `.fna` | Secuencias (sin calidad) | Genomas de referencia, transcritos, proteínas |
| FASTQ | `.fastq`, `.fq` | Secuencias + calidad por base | Lecturas crudas de un secuenciador |
| SAM/BAM | `.sam` (texto) / `.bam` (binario) | Lecturas alineadas a una referencia | Salida de un alineador (clase 4) |
| VCF | `.vcf`, `.vcf.gz` | Variantes (SNPs, indels) respecto a una referencia | Llamado de variantes (clase 6-7) |
| BED | `.bed` | Intervalos genómicos simples (coordenadas) | Regiones de interés, filtros |
| GFF/GTF | `.gff`, `.gff3`, `.gtf` | Anotación de genes/transcritos/exones | Modelos génicos, cuantificación (RNA-Seq) |

En general: **texto plano y tabular** (se puede explorar con `head`, `less`, `grep`, `awk`, `sed`, como en la clase pasada) salvo BAM, que es binario y requiere una herramienta especial (`samtools`, que instalaremos hoy).

### 2.2 FASTA: secuencias

- Cada secuencia empieza con una línea `>` (encabezado/identificador + descripción opcional).
- Las líneas siguientes son la secuencia (nucleótidos o aminoácidos), a veces partida en varias líneas.
- **No** contiene información de calidad.

In [ ]:
%%bash
cd ~/bioinfo/clase3
mkdir -p formatos && cd formatos

cat > ejemplo.fasta << 'EOF'
>gen_BRCA1 ejemplo con descripcion
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA
ATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGAC
>gen_TP53
ATGGAGGAGCCGCAGTCAGATCCTAGCGTCGAGCCCCCTCTGAGTCAGGAAACATTTTCA
EOF

echo "--- Numero de secuencias (encabezados) ---"
grep -c "^>" ejemplo.fasta

echo "--- Encabezados ---"
grep "^>" ejemplo.fasta

### 2.3 FASTQ: secuencias + calidad

Cada lectura ocupa **4 líneas**:

```
@identificador_de_la_lectura
SECUENCIA
+
CALIDAD (un caracter por base, codificación Phred+33)
```

La línea de calidad es clave: cada símbolo representa qué tan confiable es la base leída en esa posición (a mayor letra/símbolo en la tabla ASCII, mayor calidad). Herramientas como `fastqc` (que verán más adelante en el curso) resumen esta información para todo un archivo.

In [ ]:
%%bash
cd ~/bioinfo/clase3/formatos

printf '@read1\nACGTACGTAC\n+\nIIIIIIIIII\n@read2\nTTGGCCAATT\n+\n!!!!!!!!!!!\n' > ejemplo.fastq
cat ejemplo.fastq

echo "--- Numero de lecturas (encabezados @) ---"
grep -c "^@" ejemplo.fastq

### 2.4 SAM/BAM: lecturas alineadas

Cuando alineamos las lecturas de un FASTQ contra un genoma de referencia (lo que haremos en la **clase 4**), el resultado es un archivo **SAM** (texto) o su versión comprimida **BAM** (binario, mucho más liviano — típicamente 4 a 10 veces más chico).

- Líneas de encabezado empiezan con `@` (`@HD`, `@SQ` con el nombre y largo de cada cromosoma, `@PG` con el programa usado).
- Cada línea de alineamiento tiene 11 columnas obligatorias: nombre de la lectura, *flag* (bandera con información como forward/reverse), cromosoma, posición, calidad de mapeo, **CIGAR** (cómo calza la lectura contra la referencia: `M` match/mismatch, `I` inserción, `D` deleción, `S` soft-clip), y más.
- Un archivo **BAM no se puede leer con `cat`/`head`** directamente porque es binario — se necesita `samtools view`. Por eso instalaremos `samtools` hoy: lo vamos a necesitar apenas empecemos a alinear.

In [ ]:
%%bash
cd ~/bioinfo/clase3/formatos

cat > ejemplo.sam << 'EOF'
@HD	VN:1.6	SO:coordinate
@SQ	SN:chr17	LN:83257441
@PG	ID:bwa	PN:bwa	VN:0.7.17	CL:bwa mem ref.fa reads.fq
read1	0	chr17	43094650	60	60M	*	0	0	ACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGT	IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII	NM:i:0
read2	16	chr17	43094700	60	60M	*	0	0	TTGGCCAATTTTGGCCAATTTTGGCCAATTTTGGCCAATTTTGGCCAATTTTGGCCAATT	IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII	NM:i:1
EOF

echo "--- Encabezado (metadatos, lineas @) ---"
grep "^@" ejemplo.sam

echo "--- Alineamientos ---"
grep -v "^@" ejemplo.sam

### 2.5 VCF: variantes

El **VCF (Variant Call Format)** describe diferencias puntuales (SNPs, inserciones/deleciones pequeñas) entre una muestra y la referencia. Lo veremos en detalle cuando lleguemos a *llamado de variantes*, pero conviene reconocerlo desde ya:

- Líneas de metadatos empiezan con `##` (versión, descripciones de los campos `INFO`/`FORMAT`).
- Una línea `#CHROM ... ` define las columnas.
- Cada variante es una fila: cromosoma, posición, ID, alelo de referencia (`REF`), alelo alternativo (`ALT`), calidad, filtro, información adicional.

In [ ]:
%%bash
cd ~/bioinfo/clase3/formatos

cat > ejemplo.vcf << 'EOF'
##fileformat=VCFv4.2
##source=clase3_demo
##INFO=<ID=DP,Number=1,Type=Integer,Description="Read Depth">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
chr17	43094692	.	G	A	99	PASS	DP=32
chr17	43104181	.	C	T	85	PASS	DP=21
chr17	7676154	.	A	G	60	PASS	DP=18
EOF

echo "--- Metadatos ---"
grep "^##" ejemplo.vcf

echo "--- Variantes (sin encabezados) ---"
grep -v "^#" ejemplo.vcf

### 2.6 BED y GFF/GTF: coordenadas y anotación

- **BED**: formato simple, tabular, sin encabezado obligatorio. Columnas mínimas: cromosoma, inicio (0-based), fin. Se usa para marcar regiones de interés (ej. exones capturados, picos de un ChIP-seq).
- **GFF/GTF**: formato de anotación (9 columnas), más rico que BED — describe genes, transcritos y exones, con atributos clave=valor en la última columna (`gene_id`, `gene_name`, etc.). GTF es una variante de GFF muy usada en RNA-Seq para cuantificar expresión por gen/transcrito.

In [ ]:
%%bash
cd ~/bioinfo/clase3/formatos

cat > ejemplo.bed << 'EOF'
chr17	43044294	43125483	BRCA1	.	-
chr17	7661779	7687538	TP53	.	+
chr7	55019017	55211628	EGFR	.	+
EOF

echo "--- Regiones (BED) ---"
cat ejemplo.bed

echo "--- Largo de cada region (repaso de awk con aritmetica) ---"
awk 'BEGIN{OFS="\t"}{print $4, $3-$2}' ejemplo.bed

cat > ejemplo.gtf << 'EOF'
chr17	ensembl	gene	43044294	43125483	.	-	.	gene_id "ENSG00000012048"; gene_name "BRCA1";
chr17	ensembl	transcript	43044294	43125364	.	-	.	gene_id "ENSG00000012048"; transcript_id "ENST00000357654";
chr17	ensembl	exon	43044294	43045802	.	-	.	gene_id "ENSG00000012048"; transcript_id "ENST00000357654"; exon_number "1";
EOF

echo "--- Solo lineas de tipo 'gene' (repaso de awk con filtro por columna) ---"
awk '$3=="gene"' ejemplo.gtf

### 2.7 Compresión e índices

En la práctica, casi ningún archivo genómico se usa "pelado":

| Archivo | Comprimido/indexado | Índice asociado |
|---|---|---|
| `referencia.fasta` | — | `.fai` (`samtools faidx`) permite acceso rápido a cualquier región sin leer todo el archivo |
| `variantes.vcf` | `.vcf.gz` (compresión `bgzip`, distinta de `gzip` normal) | `.tbi` (`tabix`) |
| `alineamiento.bam` | ya es binario | `.bai` (`samtools index`) |

Estos índices son justamente lo que nos va a permitir, en la clase 4, tomar un BAM de varios GB y consultar solo la región que nos interesa en milisegundos, en vez de recorrer todo el archivo. Por eso instalar `samtools` hoy es el primer paso concreto hacia el alineamiento.

---
## 3. Manejo de archivos en la nube: descarga y transferencia (30 min)

Los archivos genómicos casi nunca "aparecen" en su carpeta de trabajo: hay que **descargarlos** desde un repositorio público (NCBI, Ensembl, ENA/SRA) 

### 3.1 `wget` vs `curl`

| | `wget` | `curl` |
|---|---|---|
| Uso típico | Descargar un archivo a disco | Transferir datos (descarga, pero también API REST, headers, etc.) |
| Sintaxis básica | `wget -O salida.fasta URL` | `curl -o salida.fasta URL` |
| Reintentos / descargas en segundo plano | Muy cómodo (`-c` para continuar, `-b` background) | Requiere más flags |
| Disponibilidad | No siempre viene instalado por defecto | Viene casi siempre preinstalado |

En bioinformática van a ver ambos constantemente en scripts de otras personas — más que memorizar uno, hay que saber leer los dos.

In [ ]:
%%bash
cd ~/bioinfo/clase3
mkdir -p referencia && cd referencia

# Genoma de referencia de ejemplo: PhiX174 (NC_001422.1), un genoma viral muy
# pequeño (~5.4 kb) que se usa como control en secuenciacion Illumina — ideal
# para practicar sin descargar gigabytes.
URL="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_001422.1&rettype=fasta&retmode=text"

echo "--- Intentando descargar con wget ---"
if wget -q -O phix174.fasta "$URL"; then
    echo "Descarga exitosa."
else
    echo "AVISO: no se pudo descargar (revisen su conexion a internet)."
    echo "Generamos una referencia sintetica de respaldo para poder seguir con la clase:"
    cat > phix174.fasta << 'EOF'
    >phix174_sintetico_respaldo (NO es la secuencia real, solo para practicar)
    GAGTTTTATCGCTTCCATGACGCAGAAGTTAACACTTTCGGATATTTCTGATGAGTCGA
    AAAATTATCTTGATAAAGCAGGAATTACTACTGCTTGTTTACGAATTAAATCGAAGTGG
    ACTGCTGGCGGAAAATGAGAAAATTCGACCTATCCTTGCGCAGCTCGAGAAGCTCTTAC
    EOF
fi

echo "--- Verificamos que el archivo exista y tenga contenido ---"
ls -la phix174.fasta
grep -c "^>" phix174.fasta
head -2 phix174.fasta

### 3.2 Verificar la integridad de una descarga

Cuando el archivo es grande (un genoma completo, un BAM de varios GB), una descarga puede cortarse a mitad de camino sin que el sistema avise con un error evidente. La forma estándar de confirmar que el archivo llegó completo es comparar su **hash** (`md5sum` o `sha256sum`) contra el que publica el repositorio de origen.

In [ ]:
%%bash
cd ~/bioinfo/clase3/referencia

echo "--- Hash del archivo descargado ---"
md5sum phix174.fasta
sha256sum phix174.fasta

echo
echo "En la practica: el sitio de descarga (NCBI, Ensembl, ENA) suele publicar"
echo "un archivo *.md5 o *.sha256 junto al dato. Se compara asi:"
echo '  echo "<hash_publicado>  phix174.fasta" | md5sum -c -'

---
## 4. Configuración de conda y ambientes de trabajo (55 min)

### 4.1 El problema que resuelve conda

En bioinformática van a instalar decenas de programas (alineadores, llamadores de variantes, cuantificadores...), cada uno con sus propias dependencias y versiones de Python/R/librerías del sistema. Es habitual que **dos herramientas necesiten versiones incompatibles** de una misma dependencia.

**`conda`** es un gestor de paquetes y de ambientes virtuales: permite crear "cajas" aisladas (**ambientes**) cada una con su propio Python, sus propias librerías y programas, sin que se pisen entre sí ni con el sistema. Además, un ambiente se puede **exportar** a un archivo de texto, lo que hace que un análisis sea reproducible por otra persona (o por ustedes mismos, en 8 meses más, cuando ya no se acuerden de qué versión de cada programa usaron).

### 4.2 Verificar si conda ya está instalado

In [ ]:
%%bash
if command -v conda &> /dev/null; then
    echo "conda ya esta instalado:"
    conda --version
else
    echo "conda no esta instalado en este sistema todavia."
fi

### 4.3 Instalar conda (si hace falta): Miniforge

Existen varias distribuciones de conda (Anaconda, Miniconda, Miniforge). Desde 2024, **Anaconda cambió los términos de licencia del canal `defaults`**, lo que generó restricciones para su uso institucional/académico a gran escala. Por eso, hoy la comunidad de bioinformática recomienda **Miniforge**: una instalación mínima de conda que viene preconfigurada para usar `conda-forge` (comunitario, sin esas restricciones) en vez de `defaults`.

> Si en el clúster de la UDD ya existe un módulo de conda/Anaconda instalado centralmente, probablemente les baste con cargarlo (`module load` o similar, revisen la documentación del clúster) en vez de instalar su propia copia. La instalación de abajo es para cuando quieren (o necesitan) su propio ambiente independiente.

In [ ]:
%%bash
cd ~/bioinfo/clase3
mkdir -p instaladores && cd instaladores

if command -v conda &> /dev/null; then
    echo "conda ya esta disponible, no es necesario instalar Miniforge."
else
    echo "--- Descargando el instalador de Miniforge3 (Linux x86_64) ---"
    wget -q -O Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"
    ls -la Miniforge3.sh

    echo "--- Instalando en modo silencioso (-b) en $HOME/miniforge3 ---"
    bash Miniforge3.sh -b -p "$HOME/miniforge3"

    echo "--- Activando conda para esta sesion de shell ---"
    source "$HOME/miniforge3/etc/profile.d/conda.sh"
    conda --version
fi

**Importante — por qué a veces `conda activate` "no funciona" en un notebook:** cada celda `%%bash` de este notebook abre una shell nueva e independiente (igual que vimos en la clase 2 con `export` y con `~/.bashrc`). Por eso, en un notebook, antes de usar `conda activate` hay que "cargar" conda explícitamente con `source` en la **misma celda**:

```bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mi_ambiente
```

En su sesión interactiva por SSH (fuera del notebook) esto no es necesario: basta con correr **una sola vez** `conda init bash` y abrir una terminal nueva; desde ahí `conda activate` funciona directo en cualquier sesión.

### 4.4 Configurar los canales correctos

Un **canal** es un repositorio de donde `conda` descarga paquetes. Para bioinformática, el estándar es usar el canal **bioconda** (programas de biología/genómica) apoyado en **conda-forge** (dependencias generales), y darle prioridad estricta para evitar conflictos entre versiones de un mismo paquete disponible en varios canales.

In [ ]:
%%bash
#source "$HOME/miniforge3/etc/profile.d/conda.sh" 2>/dev/null || true

conda config --add channels bioconda
conda config --add channels conda-forge
conda config --set channel_priority strict

echo "--- Canales configurados (de menor a mayor prioridad) ---"
conda config --show channels

### 4.5 Crear y activar un ambiente de trabajo

Vamos a crear un ambiente llamado `alineamiento`, que usaremos hoy y en la clase 4.

In [ ]:
%%bash
#source "$HOME/miniforge3/etc/profile.d/conda.sh"

echo "--- Creando el ambiente 'alineamiento' con Python 3.10 ---"
conda create -n alineamiento python=3.10 -y

echo "--- Ambientes disponibles ---"
conda env list

echo "--- Activando el ambiente y verificando ---"
conda activate alineamiento
python --version
which python

### 4.6 Instalar BWA y samtools dentro del ambiente

`bwa` es uno de los alineadores de lecturas cortas más usados (lo usaremos en la clase 4 para alinear un FASTQ contra una referencia). `samtools` es la herramienta estándar para manipular archivos SAM/BAM (verlos, indexarlos, filtrarlos, obtener estadísticas).

In [ ]:
%%bash
#source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate alineamiento

echo "--- Instalando bwa y samtools desde bioconda ---"
conda install -y bwa samtools

echo "--- Verificando instalacion ---"
echo "> bwa:"
bwa 2>&1 | head -5
echo
echo "> samtools:"
samtools --version | head -3

### 4.7 Buenas prácticas: listar, exportar, reproducibilidad

Antes de terminar, dos comandos que van a usar todo el semestre:

- `conda list` (con el ambiente activo): qué está instalado y en qué versión exacta.
- `conda env export > ambiente.yml`: guarda **todo** el ambiente (paquetes + versiones exactas) en un archivo de texto que pueden compartir o versionar con `git`, para que cualquiera (incluidos ustedes mismos, más adelante) pueda recrear exactamente el mismo ambiente con `conda env create -f ambiente.yml`.

In [ ]:
%%bash
#source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate alineamiento

echo "--- Paquetes instalados en el ambiente 'alineamiento' ---"
conda list | grep -E "bwa|samtools|python"

echo "--- Exportando el ambiente ---"
cd ~/bioinfo/clase3
conda env export > ambiente_alineamiento.yml
head -15 ambiente_alineamiento.yml
echo "..."

conda deactivate

---
## 5. Cierre y resumen

### Conceptos y comandos vistos hoy

```
Formatos:  FASTA, FASTQ, SAM/BAM, VCF, BED, GFF/GTF
Indices:   .fai, .bai, .tbi
Descarga:  wget, curl
Integridad: md5sum, sha256sum

conda:     conda create, activate, deactivate, install, list, env list, env export
Canales:   bioconda, conda-forge, channel_priority strict
```

### Lo que dejamos listo para la clase 4

- Un ambiente `alineamiento` con `bwa` y `samtools` instalados y verificados.
- Un genoma de referencia de ejemplo (`~/bioinfo/clase3/referencia/phix174.fasta`).
- Saben cómo se ve un archivo SAM y por qué un BAM necesita `samtools` para leerse.

### Próxima clase (Clase 4)

**Alineamiento de secuencias y procesamiento de archivos SAM/BAM.** Vamos a usar el ambiente y la referencia que dejamos listos hoy para:

1. Indexar la referencia con `bwa index`.
2. Alinear lecturas FASTQ contra la referencia con `bwa mem`.
3. Convertir, ordenar e indexar el resultado con `samtools` (`sort`, `index`).
4. Explorar el BAM resultante y sacar estadísticas básicas (`samtools flagstat`, `samtools stats`).

**Tarea para antes de la próxima clase:**

1. Verifiquen que su ambiente `alineamiento` (o el que hayan creado en el ejercicio integrador) sigue existiendo (`conda env list`) y que `bwa`/`samtools` responden correctamente.
2. Repasen la sección 2 de este notebook (tipos de archivo) — en la clase 4 vamos a dar por hecho que reconocen FASTA, FASTQ, SAM y CIGAR sin tener que volver a explicarlos.